In [2]:
import glob
import pandas as pd
from datetime import datetime, date, timedelta
from plotly import graph_objects as go

In [3]:
baseline_bt_files = glob.glob(f"/home/cloudcraftz/Office_Projects/10_Fintech/backtest_2.0_dev/HFT-Options-EIS-Global/tradelib/outputs/Condor_apr21/2024_BAJAJAUTO_STRADDLE_STRATEGY/consolidated_store/*.csv") #old

Alpha_bt_files = glob.glob(f"/home/cloudcraftz/Office_Projects/10_Fintech/backtest_2.0_dev/HFT-Options-EIS-Global/tradelib/outputs/Condor_apr21/2024_BAJAJAUTO_STRADDLE_STRATEGY_market/consolidated_store/*.csv") #new

Beta_bt_files = glob.glob(f"/home/cloudcraftz/Office_Projects/10_Fintech/backtest_2.0_dev/HFT-Options-EIS-Global/tradelib/outputs/Condor_apr21/2024_BAJAJAUTO_STRADDLE_STRATEGY_mid/consolidated_store/*.csv") #new

In [4]:
backtest_bt_df_list = [pd.read_csv(i) for i in baseline_bt_files]
Alpha_bt_df_list = [pd.read_csv(i) for i in Alpha_bt_files]
Beta_bt_df_list = [pd.read_csv(i) for i in Beta_bt_files]

In [5]:
len(backtest_bt_df_list), len(Alpha_bt_df_list), len(Beta_bt_df_list)

(240, 240, 240)

In [6]:
Alpha_bt_date = [i.split('/')[-1].split('.')[0] for i in Alpha_bt_files]
Beta_bt_date = [i.split('/')[-1].split('.')[0] for i in Beta_bt_files]
baseline_bt_date = [i.split('/')[-1].split('.')[0] for i in baseline_bt_files]

In [44]:
# for i in Alpha_bt_date:
#     if i not in baseline_bt_date:
#         print(i)

In [45]:
# for i in baseline_bt_date:
#     if i not in Alpha_bt_date:
#         print(i)

In [7]:
backtest_df = pd.concat(backtest_bt_df_list, ignore_index=True)
Alpha_df = pd.concat(Alpha_bt_df_list, ignore_index=True)
Beta_df = pd.concat(Beta_bt_df_list, ignore_index=True)

backtest_df = backtest_df[backtest_df['trade_done']==True]
Alpha_df = Alpha_df[Alpha_df['trade_done']==True]
Beta_df = Beta_df[Beta_df['trade_done']==True]

backtest_df['Timestamp'] = pd.to_datetime(backtest_df['timestamp'])
Alpha_df['Timestamp'] = pd.to_datetime(Alpha_df['timestamp'])
Beta_df['Timestamp'] = pd.to_datetime(Beta_df['timestamp'])

backtest_df = backtest_df.set_index("Timestamp")
Alpha_df = Alpha_df.set_index("Timestamp")
Beta_df = Beta_df.set_index("Timestamp")

backtest_df = backtest_df.sort_index()
Alpha_df = Alpha_df.sort_index()
Beta_df = Beta_df.sort_index()

In [8]:
frequency = "D"
backtest_daily_df = backtest_df.resample(frequency).last().dropna() #.resample("D").last().dropna()
Alpha_daily_df = Alpha_df.resample(frequency).last().dropna() #.resample("D").last().dropna()
Beta_daily_df = Beta_df.resample(frequency).last().dropna()

In [12]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

baceline_p_value = go.Scatter(x=backtest_daily_df.index, y=backtest_daily_df['portfolio_value'], mode='lines', name='Customer', line=dict(color='#6baed6'))
strangle_p_value = go.Scatter(x=backtest_daily_df.index, y=Alpha_daily_df['portfolio_value'], mode='lines', name='Market Maker', line=dict(color='#4daf4a'))
# # strangle_p_value = go.Scatter(x=backtest_daily_df.index, y=backtest_daily_df['Values']-Alpha_daily_df['portfolio_value'], mode='lines', name='diff', line=dict(color='#4daf4a'))
beta_p_value = go.Scatter(x=Beta_daily_df.index, y=Beta_daily_df['portfolio_value'], mode='lines', name='Mid', line=dict(color='#fd8d3c'))
spot_value = go.Scatter(x=backtest_daily_df.index, y=backtest_daily_df['spot'], mode='lines', name='Spot')

# Create a layout for the figure
# layout = go.Layout(title=f'Portfolio {comp_var} Comparison | Straddle + UBH + GH VS Straddle + UBH + GH(OTM/ATM) | SPXW Weekly {year}', xaxis=dict(title='Date'), yaxis=dict(title='Frequency'))

# Combine trace and layout into a figure
fig = go.Figure()
fig = make_subplots(specs = [[{"secondary_y": True}]])
# fig.add_trace(baceline_p_value, secondary_y = False)
# fig.add_trace(strangle_p_value, secondary_y = False)

# fig = go.Figure()
fig.add_trace(baceline_p_value)
fig.add_trace(strangle_p_value)
fig.add_trace(beta_p_value)
fig.add_trace(spot_value, secondary_y = True)

fig.update_layout(
   xaxis = dict(
                title='Date',
                # type="category", 
                # categoryorder='category ascending',
                # tickvals= [ts for ts in backtest_daily_df.index[::3]][::7],#list(df.index)[::50], 
                # ticktext=[ts for ts in df['Timestamp'] if ts.time() == datetime.datetime.strptime('09:20:00', '%H:%M:%S').time()], #list(pd.to_datetime(df['Timestamp']).dt.strftime('%Y-%m-%d %H:%M:%S'))[::7],
                # tickmode='array',
                # tickangle=90
            ),
    yaxis = dict(
                title = f'Portfolio Value',
                # tickvals = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
            ),

    plot_bgcolor = "white",
    title=f'Portfolio Value Comparison | Year 2024 | BAJAJ-AUTO',
)

fig.add_annotation(
                    y=backtest_daily_df['portfolio_value'][-1],
                    x=backtest_daily_df.index[-1],
                    text=f"{round(backtest_daily_df['portfolio_value'][-1]/1000)}K",
                    showarrow=False,
                    xshift=30,
                    yshift=5,
                    bgcolor='#6baed6',
                    font_color='rgb(255,255,255)',
                    font_size=17
                )

fig.add_annotation(
                    y=Alpha_daily_df['portfolio_value'][-1],
                    x=backtest_daily_df.index[-1],
                    text=f"{round(Alpha_daily_df['portfolio_value'][-1]/1000)}K",
                    showarrow=False,
                    xshift=30,
                    yshift=10,
                    bgcolor='#4daf4a',
                    font_color='rgb(255, 255, 255)',
                    font_size=17
                )

fig.add_annotation(
                    y=Beta_daily_df['portfolio_value'][-1],
                    x=backtest_daily_df.index[-1],
                    text=f"{round(Beta_daily_df['portfolio_value'][-1]/1000)}K",
                    showarrow=False,
                    xshift=30,
                    yshift=10,
                    bgcolor='#fd8d3c',
                    font_color='rgb(255, 255, 255)',
                    font_size=17
                )

fig.update_xaxes(
                mirror=True,
                ticks='outside',
                showline=True,
                linecolor='black',
                gridcolor='white')
 
fig.update_yaxes(
            mirror=True,
            ticks='outside',
            showline=True,
            linecolor='black',
            gridcolor='white')



# Show the figure
fig.show()


## Intraday

In [20]:
12345678
Cloud@123
cloud@123
1234

12345678

In [19]:
# baseline_bt_files = glob.glob(f"/home/cloudcraftz/Office_Projects/10_Fintech/backtest_2.0_dev/HFT-Options-EIS-Global/tradelib/outputs/SBIN/2024_SBIN_STRADDLE_STRATEGY_test_3/backtest/*.csv") #old

Alpha_bt_files = glob.glob(f"/home/cloudcraftz/Office_Projects/10_Fintech/backtest_2.0_dev/HFT-Options-EIS-Global/tradelib/outputs/SBIN/2024_SBIN_STRADDLE_STRATEGY_test_3/backtest/*.csv") #new

In [20]:
# backtest_bt_df_list = [pd.read_csv(i) for i in baseline_bt_files]
Alpha_bt_df_list = [pd.read_csv(i) for i in Alpha_bt_files]

In [21]:
# backtest_df = pd.concat(backtest_bt_df_list, ignore_index=True)
Alpha_df = pd.concat(Alpha_bt_df_list, ignore_index=True)

Alpha_df = Alpha_df[Alpha_df['trade_done']==True]

# backtest_df['Timestamp'] = pd.to_datetime(backtest_df['Timestamp'])
Alpha_df['Timestamp'] = pd.to_datetime(Alpha_df['timestamp'])

# backtest_df = backtest_df.set_index("Timestamp")
Alpha_df = Alpha_df.set_index("Timestamp")

# backtest_df = backtest_df.sort_index()
Alpha_df = Alpha_df.sort_index()

In [22]:
frequency = "D"
# backtest_daily_df = backtest_df.resample(frequency).last().dropna() #.resample("D").last().dropna()
Alpha_daily_df = Alpha_df.resample(frequency).last().dropna() #.resample("D").last().dropna()
# Beta_daily_df = Beta_df.resample(frequency).last().dropna()

In [23]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# baceline_p_value = go.Scatter(x=backtest_daily_df.index, y=backtest_daily_df['Values'], mode='lines', name='old', line=dict(color='#6baed6'))
strangle_p_value = go.Scatter(x=Alpha_daily_df.index, y=Alpha_daily_df['portfolio_value'], mode='lines', name='new', line=dict(color='#4daf4a'))
# strangle_p_value = go.Scatter(x=backtest_daily_df.index, y=backtest_daily_df['Values']-Alpha_daily_df['portfolio_value'], mode='lines', name='diff', line=dict(color='#4daf4a'))
# beta_p_value = go.Scatter(x=Beta_daily_df.index, y=Beta_daily_df[comp_var], mode='lines', name='Straddle+UBH', line=dict(color='#fd8d3c'))

# Create a layout for the figure
# layout = go.Layout(title=f'Portfolio {comp_var} Comparison | Straddle + UBH + GH VS Straddle + UBH + GH(OTM/ATM) | SPXW Weekly {year}', xaxis=dict(title='Date'), yaxis=dict(title='Frequency'))

# Combine trace and layout into a figure
# fig = go.Figure(layout=layout)
# fig = make_subplots(specs = [[{"secondary_y": True}]])
# fig.add_trace(baceline_p_value, secondary_y = False)
# fig.add_trace(strangle_p_value, secondary_y = False)

fig = go.Figure()
# fig.add_trace(baceline_p_value)
fig.add_trace(strangle_p_value)
# fig.add_trace(beta_p_value)

fig.update_layout(
   xaxis = dict(
                title='Date',
                # type="category", 
                # categoryorder='category ascending',
                # tickvals= [ts for ts in backtest_daily_df.index[::3]][::7],#list(df.index)[::50], 
                # ticktext=[ts for ts in df['Timestamp'] if ts.time() == datetime.datetime.strptime('09:20:00', '%H:%M:%S').time()], #list(pd.to_datetime(df['Timestamp']).dt.strftime('%Y-%m-%d %H:%M:%S'))[::7],
                # tickmode='array',
                # tickangle=90
            ),
    yaxis = dict(
                title = f'Portfolio Value',
                # tickvals = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
            ),

    plot_bgcolor = "white",
    title=f'Portfolio Value Comparison | Year 2024',
)

# fig.add_annotation(
#                     y=backtest_daily_df['Values'][-1],
#                     x=backtest_daily_df.index[-1],
#                     text=f"{round(backtest_daily_df['Values'][-1]/1000)}K",
#                     showarrow=False,
#                     xshift=30,
#                     yshift=5,
#                     bgcolor='#6baed6',
#                     font_color='rgb(255,255,255)',
#                     font_size=17
#                 )

fig.add_annotation(
                    y=Alpha_daily_df['portfolio_value'][-1],
                    x=backtest_daily_df.index[-1],
                    text=f"{round(Alpha_daily_df['portfolio_value'][-1]/1000)}K",
                    showarrow=False,
                    xshift=30,
                    yshift=10,
                    bgcolor='#4daf4a',
                    font_color='rgb(255, 255, 255)',
                    font_size=17
                )

# fig.add_annotation(
#                     y=Beta_daily_df[comp_var][-1],
#                     x=backtest_daily_df.index[-1],
#                     text=f"{round(Beta_daily_df[comp_var][-1]/1000)}K",
#                     showarrow=False,
#                     xshift=30,
#                     yshift=10,
#                     bgcolor='#fd8d3c',
#                     font_color='rgb(255, 255, 255)',
#                     font_size=17
#                 )

fig.update_xaxes(
                mirror=True,
                ticks='outside',
                showline=True,
                linecolor='black',
                gridcolor='white')
 
fig.update_yaxes(
            mirror=True,
            ticks='outside',
            showline=True,
            linecolor='black',
            gridcolor='white')



# Show the figure
fig.show()


## Day

In [55]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

def Comparison(comp_var, date_1):
    date_str = date_1.strftime("%Y%m%d")
    print(date_str)

    # backtest_df = pd.read_csv(f"/home/cloudcraftz/Downloads/SPX_Condor_Mid_2023/object_store/backtest/{date_str}.csv") #old
    # backtest_df = pd.read_csv(f"/home/cloudcraftz/Office_Projects/10_Fintech/future_intraday_22_jul/HFT-Options-EIS-Global/sig_sample/outputs/SPXW/test_condor_1/object_store/backtest/{date_str}.csv") #old
    H23_df = pd.read_csv(f"/home/cloudcraftz/Office_Projects/10_Fintech/backtest_2.0_dev/HFT-Options-EIS-Global/tradelib/outputs/SBIN/2024_BHARTIARTL_STRADDLE_STRATEGY/backtest/{date_str}.csv") #new

    H23_df = H23_df[H23_df['trade_done']==True]

    # backtest_df['Timestamp'] = pd.to_datetime(backtest_df['Timestamp'])
    H23_df['Timestamp'] = pd.to_datetime(H23_df['timestamp'])

    # backtest_df = backtest_df.set_index("Timestamp")
    H23_df = H23_df.set_index("Timestamp")

    # backtest_daily_df = backtest_df.sort_index()
    H23_daily_df = H23_df.sort_index()

    # baceline_p_value = go.Scatter(x=backtest_daily_df.index, y=backtest_daily_df["Values"], mode='lines', name="old", line=dict(color='#6baed6'))
    spot_p_value = go.Scatter(x=H23_daily_df.index, y=H23_daily_df['spot'], mode='lines', name='underlying')
    strangle_p_value = go.Scatter(x=H23_daily_df.index, y=H23_daily_df["portfolio_value"], mode='lines', name='New', line=dict(color='#4daf4a'))

    fig = go.Figure()
    fig = make_subplots(specs = [[{"secondary_y": True}]])

    # fig.add_trace(baceline_p_value)
    fig.add_trace(strangle_p_value)
    fig.add_trace(spot_p_value, secondary_y = True)

    fig.update_layout(
    xaxis = dict(
                    title='Date',
                ),
        yaxis = dict(
                    title = f'Portfolio {comp_var}',
                ),

        plot_bgcolor = "white",
        title=f'Minute wise {comp_var} | BHARTIARTL {date_1.year} | date {date_1.date()}',
    )

    fig.update_xaxes(
                    mirror=True,
                    ticks='outside',
                    showline=True,
                    linecolor='black',
                    gridcolor='white')
    
    fig.update_yaxes(
                mirror=True,
                ticks='outside',
                showline=True,
                linecolor='black',
                gridcolor='white')

    # fig.update_yaxes(title_text = f"underlying", secondary_y = True)

    fig.show()


In [57]:
comp_var = "Values"
a = datetime(2024, 6, 27)
Comparison(comp_var, a)

20240627


In [53]:
comp_var = "Values"
a = datetime(2024, 8, 29)
Comparison(comp_var, a)

20240829


In [39]:
comp_var = "Values"
a = datetime(2024, 6, 27)
Comparison(comp_var, a)

20240627


## Daily IV Changed

In [15]:
baseline_bt_files = glob.glob(f"/home/cloudcraftz/Office_Projects/10_Fintech/backtest_2.0_dev/HFT-Options-EIS-Global/tradelib/outputs/SBIN/2024_SBIN_STRADDLE_STRATEGY_test_4/consolidated_store/*.csv")
backtest_bt_df_list = [pd.read_csv(i) for i in baseline_bt_files]

In [16]:
backtest_df = pd.concat(backtest_bt_df_list, ignore_index=True)

backtest_df = backtest_df[backtest_df['trade_done']==True]

backtest_df['Timestamp'] = pd.to_datetime(backtest_df['timestamp'])
backtest_df = backtest_df.set_index("Timestamp")
backtest_df = backtest_df.sort_index()

In [17]:
# Define target times
target_times = ['09:20:00', '15:15:00']

# Extract IV at 9:20 AM and 3:15 PM
backtest_df['Time'] = backtest_df.index.time  # Extract time from index
daily_iv = backtest_df[backtest_df['Time'].astype(str).isin(target_times)].copy()
daily_iv['Date'] = daily_iv.index.date  # Extract date for grouping
daily_iv = daily_iv.pivot(index='Date', columns='Time', values='atm_iv')  # Reshape for clarity
daily_iv.columns = ['IV_09:20', 'IV_15:15']  # Rename columns

# Calculate daily IV change (IV at 15:15 - IV at 09:20)
daily_iv['IV_Change'] = (daily_iv['IV_15:15']/daily_iv['IV_09:20']) -1


In [18]:
import plotly.express as px

# Create a line chart for IV_Change
fig = px.line(daily_iv, x=daily_iv.index, y='IV_Change', title='Daily IV Change Over Time', 
              labels={'Date': 'Date', 'IV_Change': 'IV Change (%)'}, markers=True)

# Show the plot
fig.show()

In [19]:
# Create a bar chart for IV_Change
fig = px.bar(daily_iv, x=daily_iv.index, y='IV_Change', title='Daily IV Change (Bar Chart)', 
             labels={'Date': 'Date', 'IV_Change': 'IV_Change'}, 
             text=round(daily_iv['IV_Change'], 2))  # Display values on bars
            #  color=daily_iv_sprade_df['IV_RV_Spread'],  # Color based on value
            #  color_continuous_scale='Bluered')

# Show the plot
fig.show()

In [20]:
backtest_df['IV_RV_Spread'] = backtest_df['atm_iv'] - backtest_df['RV']
not_null_backtest_df = backtest_df.dropna(axis=0)
daily_iv_sprade_df = not_null_backtest_df.resample('D')[['IV_RV_Spread']].mean().dropna()

In [21]:
# Create a bar chart for IV_Change
fig = px.bar(daily_iv_sprade_df, x=daily_iv_sprade_df.index, y='IV_RV_Spread', title='Average Daily IV_RV_Spread', 
             labels={'Date': 'Date', 'IV_RV_Spread': 'IV_RV_Spread'}, 
             text=round(daily_iv_sprade_df['IV_RV_Spread'], 2))  # Display values on bars
            #  color=daily_iv_sprade_df['IV_RV_Spread'],  # Color based on value
            #  color_continuous_scale='Bluered')

# Show the plot
fig.show()


In [22]:
daily_iv_sprade_df['IV_RV_Spread'].describe()

count    241.000000
mean       0.037982
std        0.071188
min       -0.612971
25%        0.009363
50%        0.042526
75%        0.072707
max        0.221037
Name: IV_RV_Spread, dtype: float64

In [23]:
daily_pnl = backtest_df.resample('D').last().dropna()
daily_pnl['daily_pnl'] = daily_pnl['portfolio_value'].diff()
daily_pnl.loc['2024-01-25']['daily_pnl']

-8737.499999999942